# Evals & Optimizer

Evaluate the single-call `CaltrainScheduleHelper` exactly as the bot runs it, then optimize its
instructions with GEPA for the fastest configuration: Claude Haiku 5.5 with reasoning turned off.

Shape: load data → build program → baseline eval → GEPA → save → re-evaluate the saved state.

Run it from the repo root with `OPENROUTER_API_KEY` available:

```
uv run --env-file .env jupyter nbconvert --to notebook --execute --inplace \
    "evals/Evals & Optimizer.ipynb" --ExecutePreprocessor.timeout=-1
```

Every cell that calls an LLM costs money. The whole notebook is sized to stay under about $3 and GEPA
stops itself early when it reaches its own spending limit (see **Settings**).

In [1]:
import contextlib
import io
import json
import logging
import os
import random
import statistics
import time
import urllib.request
from datetime import datetime, timedelta
from pathlib import Path

import dspy
from dspy.teleprompt import GEPA
from dspy.teleprompt.gepa.gepa_utils import DSPyTrace
from gepa.strategies.instruction_proposal import InstructionProposalSignature
from loguru import logger

from caltrain_bot.config import OpenRouterSettings
from caltrain_bot.question_analysis import (
    CALTRAIN_TIMEZONE,
    CaltrainScheduleHelper,
    ScheduleQuestion,
    UnsupportedQuestion,
    build_lm,
)

In [2]:
# The program logs every analysis; keep the notebook output readable.
logger.remove()

# Settings

In [3]:
STUDENT_MODEL = "anthropic/claude-haiku-5.5"
STUDENT_REASONING_EFFORT = (
    "none"  # the optimization target: no reasoning is the fastest, cheapest setup
)
REFLECTION_MODEL = "anthropic/claude-sonnet-5.5"

# USD per million tokens (input, output), from https://openrouter.ai/api/v1/models
PRICES = {
    STUDENT_MODEL: (0.10, 0.50),
    REFLECTION_MODEL: (2.00, 10.00),
}

GEPA_MAX_METRIC_CALLS = 1500
GEPA_REFLECTION_MINIBATCH_SIZE = 6
# GEPA does not start another iteration once its own LLM calls have cost this much.
GEPA_SPEND_LIMIT_USD = 2.00

PROGRAM_PATH = Path("../data/schedule_question_program.json")
NUM_THREADS = 8
LATENCY_SAMPLE_SIZE = 30

Spending is tracked two ways. OpenRouter reports the cost of every call in its response, so the sum over
the notebook's own calls is exact and immediate; that is what the limits use. The key's cumulative usage
is read at the start and the end as a cross-check: it lags by up to a minute and includes anything else
that uses the same key while the notebook runs.

In [4]:
def openrouter_key_usage() -> float:
    """Cumulative USD spent by the API key."""
    request = urllib.request.Request(
        "https://openrouter.ai/api/v1/key",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return float(json.load(response)["data"]["usage"])


tracked_lms: list[dspy.LM] = []


def spent_usd() -> float:
    """USD charged for this notebook's own LLM calls, as reported by OpenRouter per call."""
    return sum(
        entry.get("cost") or 0.0 for used_lm in tracked_lms for entry in used_lm.history
    )


USAGE_AT_START = openrouter_key_usage()
spend_log: dict[str, float] = {}
print(f"Key usage at start: ${USAGE_AT_START:.4f}")

Key usage at start: $7.7832


# Data

In [5]:
def load_jsonl(
    path: str | Path,
    *,
    input_keys: tuple[str, ...] = ("question", "reference_datetime"),
) -> list[dspy.Example]:
    examples = []

    with Path(path).open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                row = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_number}"
                ) from error

            example = dspy.Example(**row).with_inputs(*input_keys)
            examples.append(example)

    return examples

In [6]:
trainset = load_jsonl("../data/train.jsonl")
valset = load_jsonl("../data/validation.jsonl")

In [7]:
print(trainset[0])

Example({'id': 'train-001', 'family_id': 'train-now-001', 'scenario': 'schedule_now_alias', 'question': 'Next Caltrain from Sunnyvale to SF?', 'reference_datetime': '2026-08-24T15:00:00-07:00', 'is_schedule_question': True, 'departure_station': 'sunnyvale', 'arrival_station': 'san francisco', 'departure_time': '2026-08-24T15:00:00-07:00', 'time_tolerance_minutes': 5}) (input_keys={'reference_datetime', 'question'})


In [8]:
for name, dataset in (("train", trainset), ("validation", valset)):
    supported = sum(example.is_schedule_question for example in dataset)
    print(
        f"{name}: {len(dataset)} questions, {supported} supported, {len(dataset) - supported} unsupported"
    )

train: 88 questions, 56 supported, 32 unsupported
validation: 91 questions, 60 supported, 31 unsupported


# Program

In [9]:
lm = build_lm(
    OpenRouterSettings(
        api_key=os.environ["OPENROUTER_API_KEY"],
        model=STUDENT_MODEL,
        reasoning_effort=STUDENT_REASONING_EFFORT,
    )
)
dspy.configure(lm=lm)
tracked_lms.append(lm)

In [10]:
stations = (
    "22nd street",
    "bayshore",
    "belmont",
    "blossom hill",
    "broadway",
    "burlingame",
    "california avenue",
    "capitol",
    "college park",
    "gilroy",
    "hayward park",
    "hillsdale",
    "lawrence",
    "menlo park",
    "millbrae",
    "morgan hill",
    "mountain view",
    "palo alto",
    "redwood city",
    "san antonio",
    "san bruno",
    "san carlos",
    "san francisco",
    "san jose diridon",
    "san martin",
    "san mateo",
    "santa clara",
    "south san francisco",
    "sunnyvale",
    "tamien",
)

In [11]:
prog = CaltrainScheduleHelper(stations)

In [12]:
prog(
    question="caltrain from sf to palo alto",
    reference_datetime=datetime.now(CALTRAIN_TIMEZONE).isoformat(),
)

ScheduleQuestion(departure_station='san francisco', arrival_station='palo alto', departure_time=datetime.datetime(2026, 11, 17, 21, 57, 46, 78882, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=57600))))

# Metric

Scoring is unchanged: an unsupported question must come back as `UnsupportedQuestion`; a supported one
must match both stations and land within `time_tolerance_minutes` of the expected departure time, with
1/3 partial credit per correct field. `gepa_metric` returns the same score plus textual feedback for
GEPA's reflection step.

In [13]:
FIELD_REWARD = 1.0 / 3.0


def parse_datetime(value: object) -> datetime | None:
    if isinstance(value, datetime):
        parsed = value
    elif isinstance(value, str):
        try:
            parsed = datetime.fromisoformat(
                value.removesuffix("Z") + ("+00:00" if value.endswith("Z") else "")
            )
        except ValueError:
            return None
    else:
        return None

    return parsed if parsed.utcoffset() is not None else None


def stations_match(expected: object, predicted: object) -> bool:
    def normalize(value: object) -> str | None:
        if not isinstance(value, str):
            return None
        return " ".join(value.casefold().split()) or None

    expected = normalize(expected)
    predicted = normalize(predicted)

    return expected is not None and predicted == expected


def times_match(
    expected: object,
    predicted: object,
    tolerance_minutes: float,
) -> bool:
    expected_time = parse_datetime(expected)
    predicted_time = parse_datetime(predicted)

    if expected_time is None or predicted_time is None:
        return False

    tolerance = timedelta(minutes=tolerance_minutes)
    return abs(predicted_time - expected_time) <= tolerance


def field_results(
    gold: dspy.Example, pred: ScheduleQuestion
) -> tuple[bool, bool, bool]:
    """Whether the departure station, arrival station, and departure time are correct."""
    return (
        stations_match(gold.departure_station, pred.departure_station),
        stations_match(gold.arrival_station, pred.arrival_station),
        times_match(
            gold.departure_time,
            pred.departure_time,  # a tz-aware datetime
            gold.time_tolerance_minutes,
        ),
    )


def baseline_metric(
    gold: dspy.Example,
    pred: ScheduleQuestion | UnsupportedQuestion,
    trace: DSPyTrace | None = None,
    pred_name: str | None = None,
    pred_trace: DSPyTrace | None = None,
) -> float:
    if not gold.is_schedule_question:
        return float(isinstance(pred, UnsupportedQuestion))

    if not isinstance(pred, ScheduleQuestion):
        return 0.0

    results = field_results(gold, pred)
    if all(results):
        return 1.0

    return sum(FIELD_REWARD for is_correct in results if is_correct)

In [14]:
def describe_time(value: object) -> str:
    """Render a time as the local wall-clock value the model is asked to produce."""
    parsed = parse_datetime(value)
    if parsed is None:
        return repr(value)
    return parsed.astimezone(CALTRAIN_TIMEZONE).strftime("%A %Y-%m-%dT%H:%M")


def describe_expected(gold: dspy.Example) -> str:
    if not gold.is_schedule_question:
        return "is_schedule_question=False with every other output null"

    time_note = ""
    if parse_datetime(gold.departure_time) == parse_datetime(gold.reference_datetime):
        time_note = (
            " (equal to reference_datetime: the message states no departure time,"
            " so the departure_time output should be null)"
        )
    return (
        f"is_schedule_question=True, departure_station={gold.departure_station!r}, "
        f"arrival_station={gold.arrival_station!r}, "
        f"departure_time={describe_time(gold.departure_time)}{time_note}"
    )


def describe_prediction(pred: object) -> str:
    if isinstance(pred, UnsupportedQuestion):
        return "rejected as unsupported"
    if isinstance(pred, ScheduleQuestion):
        return (
            f"departure_station={pred.departure_station!r}, "
            f"arrival_station={pred.arrival_station!r}, "
            f"departure_time={describe_time(pred.departure_time)}"
        )
    return "no answer (the LLM call failed or its output could not be parsed)"


def build_feedback(
    gold: dspy.Example, pred: object, score: float, trace: DSPyTrace | None = None
) -> str:
    context = (
        f"Dataset scenario: {gold.scenario}. "
        f"reference_datetime: {describe_time(gold.reference_datetime)}. "
        f"Expected: {describe_expected(gold)}."
    )
    if score == 1.0:
        return f"Correct. {context}"

    if not isinstance(pred, ScheduleQuestion | UnsupportedQuestion):
        return f"Incorrect: {describe_prediction(pred)}. {context}"

    if not gold.is_schedule_question:
        return (
            "Incorrect: this message is NOT a supported schedule lookup, but it was answered "
            f"as one ({describe_prediction(pred)}). {context}"
        )

    if isinstance(pred, UnsupportedQuestion):
        raw_output = trace[-1][2] if trace else None
        if getattr(raw_output, "is_schedule_question", False):
            reason = (
                "its answer (is_schedule_question=True) was discarded because "
                "a station was null or not a known station, both stations were the same, or "
                "departure_time was not a valid YYYY-MM-DDTHH:MM value"
            )
        else:
            reason = "it was rejected as unsupported (is_schedule_question=False)"
        return f"Incorrect: this message IS a supported schedule lookup, but {reason}. {context}"

    departure_ok, arrival_ok, time_ok = field_results(gold, pred)
    problems = []
    if not departure_ok:
        problems.append(
            f"departure_station should be {gold.departure_station!r}, got {pred.departure_station!r}"
        )
    if not arrival_ok:
        problems.append(
            f"arrival_station should be {gold.arrival_station!r}, got {pred.arrival_station!r}"
        )
    if not time_ok:
        expected_time = parse_datetime(gold.departure_time)
        minutes_off = round((pred.departure_time - expected_time).total_seconds() / 60)
        problems.append(
            f"departure_time should be {describe_time(expected_time)} "
            f"(within {gold.time_tolerance_minutes} minutes), got {describe_time(pred.departure_time)}, "
            f"which is {abs(minutes_off)} minutes too {'late' if minutes_off > 0 else 'early'}"
        )
    return f"Partly incorrect (score {score:.2f}): {'; '.join(problems)}. {context}"


def gepa_metric(
    gold: dspy.Example,
    pred: ScheduleQuestion | UnsupportedQuestion,
    trace: DSPyTrace | None = None,
    pred_name: str | None = None,
    pred_trace: DSPyTrace | None = None,
) -> dspy.Prediction:
    score = baseline_metric(gold, pred)
    return dspy.Prediction(
        score=score, feedback=build_feedback(gold, pred, score, trace)
    )

In [15]:
# What the feedback looks like for a correct answer and for the two kinds of rejection mistakes.
supported_example = next(
    example for example in trainset if example.is_schedule_question
)
unsupported_example = next(
    example for example in trainset if not example.is_schedule_question
)
correct_answer = ScheduleQuestion(
    departure_station=supported_example.departure_station,
    arrival_station=supported_example.arrival_station,
    departure_time=parse_datetime(supported_example.departure_time),
)
for gold, pred in (
    (supported_example, correct_answer),
    (supported_example, UnsupportedQuestion()),
    (unsupported_example, correct_answer),
):
    result = gepa_metric(gold, pred)
    print(f"{result.score:.2f}  {gold.question!r}\n      {result.feedback}\n")

1.00  'Next Caltrain from Sunnyvale to SF?'
      Correct. Dataset scenario: schedule_now_alias. reference_datetime: Monday 2026-08-24T15:00. Expected: is_schedule_question=True, departure_station='sunnyvale', arrival_station='san francisco', departure_time=Monday 2026-08-24T15:00 (equal to reference_datetime: the message states no departure time, so the departure_time output should be null).

0.00  'Next Caltrain from Sunnyvale to SF?'
      Incorrect: this message IS a supported schedule lookup, but it was rejected as unsupported (is_schedule_question=False). Dataset scenario: schedule_now_alias. reference_datetime: Monday 2026-08-24T15:00. Expected: is_schedule_question=True, departure_station='sunnyvale', arrival_station='san francisco', departure_time=Monday 2026-08-24T15:00 (equal to reference_datetime: the message states no departure time, so the departure_time output should be null).

0.00  'How many bikes are allowed in the bike car?'
      Incorrect: this message is NOT a sup

# Evaluation helpers

GEPA reflects on the train set and selects candidates on the validation set, so the two are reported
separately as well as merged. There is no untouched test set: after optimization the train score is
biased by reflection and the validation score by selection.

Latency is timed on a sequential pass, because threaded evaluation distorts per-question timings.

In [16]:
def evaluate(program: dspy.Module, label: str) -> dict:
    """Score a program on train and validation separately and merged; list every miss."""
    report = {"label": label, "misses": [], "errors": 0}
    scores_by_split = {}

    for name, dataset in (("train", trainset), ("validation", valset)):
        evaluator = dspy.Evaluate(
            devset=dataset,
            metric=baseline_metric,
            num_threads=NUM_THREADS,
            display_progress=False,
            display_table=False,
            provide_traceback=False,
            max_errors=len(dataset),
            failure_score=0.0,
        )
        results = evaluator(program).results
        scores_by_split[name] = [score for _, _, score in results]
        for example, prediction, score in results:
            if score < 1.0:
                report["misses"].append((name, example, prediction, score))
            if not isinstance(prediction, ScheduleQuestion | UnsupportedQuestion):
                report["errors"] += 1

    scores_by_split["merged"] = scores_by_split["train"] + scores_by_split["validation"]
    for name, scores in scores_by_split.items():
        report[name] = 100 * sum(scores) / len(scores)
        report[f"{name}_perfect"] = sum(score == 1.0 for score in scores)
        report[f"{name}_total"] = len(scores)

    print(f"== {label} ==")
    for name in ("train", "validation", "merged"):
        print(
            f"{name:>10}: {report[name]:5.1f}%  "
            f"({report[f'{name}_perfect']}/{report[f'{name}_total']} fully correct)"
        )
    print(f"LLM/parse errors: {report['errors']}; misses: {len(report['misses'])}")
    for name, example, prediction, score in report["misses"]:
        print(
            f"  MISS {score:.2f} [{name} / {example.scenario}] {example.question!r}\n"
            f"       expected:  {describe_expected(example)}\n"
            f"       predicted: {describe_prediction(prediction)}"
        )
    return report


latency_sample = random.Random(0).sample(trainset + valset, LATENCY_SAMPLE_SIZE)


def measure_latency(program: dspy.Module, label: str) -> dict:
    """Time a sequential pass over a fixed sample and report token usage per call."""
    calls_before = len(lm.history)
    latencies = []
    for example in latency_sample:
        started = time.perf_counter()
        try:
            program(**example.inputs())
        except Exception as error:  # a failed call still took this long for the user
            print(
                f"  call failed after {time.perf_counter() - started:.1f}s: {error!r:.200}"
            )
        latencies.append(time.perf_counter() - started)

    usage = [entry.get("usage") or {} for entry in lm.history[calls_before:]]
    report = {
        "median_s": statistics.median(latencies),
        "mean_s": statistics.mean(latencies),
        "max_s": max(latencies),
        "prompt_tokens": statistics.mean(u.get("prompt_tokens", 0) for u in usage),
        "completion_tokens": statistics.mean(
            u.get("completion_tokens", 0) for u in usage
        ),
    }
    print(
        f"{label}: {len(latencies)} sequential questions, latency median {report['median_s']:.2f}s, "
        f"mean {report['mean_s']:.2f}s, max {report['max_s']:.2f}s; "
        f"avg tokens per call: prompt {report['prompt_tokens']:.0f}, "
        f"completion {report['completion_tokens']:.0f}"
    )
    return report

# Baseline

The unoptimized program with reasoning off. It is evaluated twice because the model is sampled at the
provider's default temperature, so scores move a little between runs.

In [17]:
baseline_runs = [evaluate(prog, f"baseline, run {run}") for run in (1, 2)]

2026/11/17 21:57:56 INFO dspy.evaluate.evaluate: Average Metric: 82.0 / 88 (93.2%)
2026/11/17 21:58:06 INFO dspy.evaluate.evaluate: Average Metric: 87.0 / 91 (95.6%)


== baseline, run 1 ==
     train:  93.2%  (82/88 fully correct)
validation:  95.6%  (87/91 fully correct)
    merged:  94.4%  (169/179 fully correct)
LLM/parse errors: 0; misses: 10
  MISS 0.00 [train / schedule_now_reversed_alias] "I need to get to 4th and King from Palo Alto. What's the next one?"
       expected:  is_schedule_question=True, departure_station='palo alto', arrival_station='san francisco', departure_time=Monday 2026-08-24T15:00 (equal to reference_datetime: the message states no departure time, so the departure_time output should be null)
       predicted: rejected as unsupported
  MISS 0.00 [train / schedule_now_alias] 'Catching the next train at Diridon going to Mountain View.'
       expected:  is_schedule_question=True, departure_station='san jose diridon', arrival_station='mountain view', departure_time=Monday 2026-08-24T15:00 (equal to reference_datetime: the message states no departure time, so the departure_time output should be null)
       predicted: rejected

2026/11/17 21:58:15 INFO dspy.evaluate.evaluate: Average Metric: 84.0 / 88 (95.5%)
2026/11/17 21:58:25 INFO dspy.evaluate.evaluate: Average Metric: 86.0 / 91 (94.5%)


== baseline, run 2 ==
     train:  95.5%  (84/88 fully correct)
validation:  94.5%  (86/91 fully correct)
    merged:  95.0%  (170/179 fully correct)
LLM/parse errors: 0; misses: 9
  MISS 0.00 [train / schedule_now_reversed_alias] "I need to get to 4th and King from Palo Alto. What's the next one?"
       expected:  is_schedule_question=True, departure_station='palo alto', arrival_station='san francisco', departure_time=Monday 2026-08-24T15:00 (equal to reference_datetime: the message states no departure time, so the departure_time output should be null)
       predicted: rejected as unsupported
  MISS 0.00 [train / schedule_now_alias] 'Catching the next train at Diridon going to Mountain View.'
       expected:  is_schedule_question=True, departure_station='san jose diridon', arrival_station='mountain view', departure_time=Monday 2026-08-24T15:00 (equal to reference_datetime: the message states no departure time, so the departure_time output should be null)
       predicted: rejected 

In [18]:
baseline_latency = measure_latency(prog, "baseline")

baseline: 30 sequential questions, latency median 0.86s, mean 0.87s, max 1.05s; avg tokens per call: prompt 3278, completion 96


In [19]:
spend_log["baseline"] = spent_usd()
print(f"Spent so far: ${spend_log['baseline']:.4f}")

Spent so far: $0.1459


# Optimizer

GEPA rewrites the instructions of the program's single predictor. Each iteration runs a minibatch of
train questions; if any is wrong, the reflection model reads the inputs, outputs, and metric feedback
and proposes new instructions, which are kept and scored on the whole validation set only when they do
better on that minibatch.

The reflection prompt is customized for this task: the student answers without reasoning, so it needs
explicit rules, and the prompt must stay short because prompt length is latency and cost on every
question.

In [20]:
reflection_lm = dspy.LM(
    f"openrouter/{REFLECTION_MODEL}",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=1.0,
    max_tokens=16000,
    num_retries=3,
    cache=False,
    extra_body={"reasoning": {"effort": "medium"}},
)
tracked_lms.append(reflection_lm)

REFLECTION_PROMPT = """I gave an assistant the instructions below. The assistant parses messages sent to a Caltrain schedule bot.
The assistant is a small, fast language model that answers in a single pass with no reasoning step. It follows explicit rules and short example phrases well, but it cannot work through long or subtle procedures.

Current instructions:
```
<curr_instructions>
```

Examples of task inputs, the assistant's outputs, and feedback from an automatic grader that knows the correct answer:
```
<inputs_outputs_feedback>
```

Write an improved version of the instructions that fixes the mistakes described in the feedback.
- Work out the general rule behind each mistake and state it as an explicit rule, with a short example phrase where that helps. Do not copy whole example messages or hard-code their answers.
- Keep every existing rule that the feedback does not contradict. The instructions cover many cases that are not shown here, and dropping or loosening a rule will break them.
- When a valid lookup was rejected, or an unsupported message was accepted, sharpen the boundary between the two without moving it for other kinds of messages.
- Keep the output rules intact: the meaning of each field, the YYYY-MM-DDTHH:MM time format, and null outputs for unsupported messages.
- Keep the instructions concise, at most about 6,000 characters. Every extra sentence slows the assistant down.

Provide the new instructions within ``` blocks."""


def propose_instructions(
    candidate: dict[str, str],
    reflective_dataset: dict[str, list[dict]],
    components_to_update: list[str],
) -> dict[str, str]:
    def ask_reflection_lm(prompt: str) -> str:
        output = reflection_lm(prompt)[0]
        return output["text"] if isinstance(output, dict) else output

    return {
        name: InstructionProposalSignature.run(
            lm=ask_reflection_lm,
            input_dict={
                "current_instruction_doc": candidate[name],
                "dataset_with_feedback": reflective_dataset[name],
                "prompt_template": REFLECTION_PROMPT,
            },
        )["new_instruction"]
        for name in components_to_update
    }

### Cost estimate

A student call costs a small fraction of a cent; a reflection call costs a few cents. An iteration whose
minibatch is all correct is skipped without reflection. A proposal that beats its parent on the minibatch
is accepted and scored on the whole validation set; a rejected one costs only the minibatch re-run, so
the fewer proposals are accepted, the more reflections fit in the metric-call budget. The estimate
assumes half are accepted. `GEPA_SPEND_LIMIT_USD` is the hard stop whatever happens.

In [21]:
def call_cost_usd(model: str, prompt_tokens: float, completion_tokens: float) -> float:
    input_price, output_price = PRICES[model]
    return (prompt_tokens * input_price + completion_tokens * output_price) / 1e6


# Prompts tend to grow during optimization, so price student calls at 1.5x the baseline prompt.
student_call_usd = call_cost_usd(
    STUDENT_MODEL,
    1.5 * baseline_latency["prompt_tokens"],
    baseline_latency["completion_tokens"],
)
# A reflection reads the instructions plus a minibatch (~5k tokens) and writes new instructions
# after some reasoning (~4k tokens).
reflection_call_usd = call_cost_usd(REFLECTION_MODEL, 5_000, 4_000)
ASSUMED_ACCEPTANCE_RATE = 0.5

baseline_train_accuracy = statistics.mean(
    run["train_perfect"] / run["train_total"] for run in baseline_runs
)
minibatch_has_miss = 1 - baseline_train_accuracy**GEPA_REFLECTION_MINIBATCH_SIZE
calls_per_reflection = (
    GEPA_REFLECTION_MINIBATCH_SIZE
    / max(minibatch_has_miss, 0.05)  # minibatches until one has a miss
    + GEPA_REFLECTION_MINIBATCH_SIZE  # re-run of the minibatch with the new instructions
    + ASSUMED_ACCEPTANCE_RATE
    * len(valset)  # full validation pass for an accepted candidate
)
estimated_reflections = (GEPA_MAX_METRIC_CALLS - len(valset)) / calls_per_reflection
estimated_gepa_usd = (
    GEPA_MAX_METRIC_CALLS * student_call_usd
    + estimated_reflections * reflection_call_usd
)
print(
    f"Student call ~${student_call_usd:.5f}, reflection call ~${reflection_call_usd:.3f}\n"
    f"Estimated reflections: {estimated_reflections:.0f} "
    f"(one per ~{calls_per_reflection:.0f} metric calls)\n"
    f"Estimated GEPA cost: ${estimated_gepa_usd:.2f} "
    f"(hard stop at ${GEPA_SPEND_LIMIT_USD:.2f})"
)
assert estimated_gepa_usd <= 2.50, (
    "GEPA run is sized above the budget; lower GEPA_MAX_METRIC_CALLS."
)

Student call ~$0.00054, reflection call ~$0.050
Estimated reflections: 20 (one per ~72 metric calls)
Estimated GEPA cost: $1.79 (hard stop at $2.00)


In [22]:
class SpendStopper:
    """Stop GEPA before a new iteration once the run has cost `limit_usd`."""

    def __init__(self, limit_usd: float) -> None:
        self.limit_usd = limit_usd
        self.spent_at_start = spent_usd()
        self.triggered = False

    def __call__(self, gepa_state: object) -> bool:
        if spent_usd() - self.spent_at_start >= self.limit_usd:
            self.triggered = True
        return self.triggered

In [23]:
class DropPerExampleDumps(logging.Filter):
    """Hide GEPA's per-example score dumps; the per-candidate summary lines are kept."""

    NOISY = ("Individual valset scores", "pareto front scores", "pareto front programs")

    def filter(self, record: logging.LogRecord) -> bool:
        return not any(text in record.getMessage() for text in self.NOISY)


logging.getLogger("dspy.teleprompt.gepa.gepa").addFilter(DropPerExampleDumps())

spend_stopper = SpendStopper(GEPA_SPEND_LIMIT_USD)
gepa_optimizer = GEPA(
    metric=gepa_metric,
    max_metric_calls=GEPA_MAX_METRIC_CALLS,
    reflection_lm=reflection_lm,
    instruction_proposer=propose_instructions,
    reflection_minibatch_size=GEPA_REFLECTION_MINIBATCH_SIZE,
    use_merge=False,  # merging recombines predictors; this program has only one
    num_threads=NUM_THREADS,
    track_stats=True,
    log_dir=os.environ.get(
        "GEPA_LOG_DIR"
    ),  # optional: checkpoints, and resumes an interrupted run
    seed=0,
    gepa_kwargs={"stop_callbacks": [spend_stopper]},
)

gepa_started = time.perf_counter()
# DSPy prints a progress bar to stdout for every minibatch; GEPA's own log (stderr) is kept.
with contextlib.redirect_stdout(io.StringIO()):
    gepa_prog = gepa_optimizer.compile(student=prog, trainset=trainset, valset=valset)
gepa_wall_seconds = time.perf_counter() - gepa_started

2026/11/17 21:58:51 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 1500 metric calls of the program. This amounts to 8.38 full evals on the train+val set.
2026/11/17 21:58:51 INFO dspy.teleprompt.gepa.gepa: Using 91 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/1500 [00:00<?, ?rollouts/s]2026/11/17 21:59:01 INFO dspy.evaluate.evaluate: Average Metric: 87.0 / 91 (95.6%)
2026/11/17 21:59:01 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.9560439560439561 over 91 / 91 examples
GEPA Optimization:   6%|▌         | 91/1500 [00:09<02:28,  9.49rollouts/s]2026/11/17 21:59:01 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 sc

Save the optimized prompt right away, as JSON state (not a pickle). This is the file
`build_caltrain_schedule_helper` loads when it exists.

In [24]:
gepa_prog.save(str(PROGRAM_PATH))
print(f"Saved {PROGRAM_PATH} ({PROGRAM_PATH.stat().st_size} bytes)")

Saved ../data/schedule_question_program.json (5400 bytes)


In [25]:
spend_log["gepa"] = spent_usd()
gepa_results = gepa_prog.detailed_results
gepa_usd = spend_log["gepa"] - spend_log["baseline"]

print(f"GEPA wall time: {gepa_wall_seconds / 60:.1f} min")
print(
    f"Metric calls used: {gepa_results.total_metric_calls} of {GEPA_MAX_METRIC_CALLS}"
)
print(f"Stopped by spending limit: {spend_stopper.triggered}")
print(f"Reflection calls ({REFLECTION_MODEL}): {len(reflection_lm.history)}")
print(f"Full validation evaluations: {gepa_results.num_full_val_evals}")
print(f"GEPA cost: ${gepa_usd:.4f}; spent so far: ${spend_log['gepa']:.4f}")
print("Validation score of each candidate (0 is the unoptimized program):")
for index, score in enumerate(gepa_results.val_aggregate_scores):
    marker = "  <- selected" if index == gepa_results.best_idx else ""
    print(
        f"  candidate {index}: {100 * score:5.1f}%  "
        f"({len(gepa_results.candidates[index].analyze.signature.instructions)} chars){marker}"
    )

GEPA wall time: 4.9 min
Metric calls used: 1505 of 1500
Stopped by spending limit: False
Reflection calls (anthropic/claude-sonnet-5.5): 12
Full validation evaluations: 11
GEPA cost: $0.9304; spent so far: $1.0763
Validation score of each candidate (0 is the unoptimized program):
  candidate 0:  95.6%  (2570 chars)
  candidate 1:  94.5%  (3579 chars)
  candidate 2:  95.6%  (3193 chars)
  candidate 3:  98.9%  (3977 chars)  <- selected
  candidate 4:  97.8%  (4349 chars)
  candidate 5:  93.4%  (4365 chars)
  candidate 6:  96.7%  (4353 chars)
  candidate 7:  95.6%  (4348 chars)
  candidate 8:  95.6%  (4357 chars)
  candidate 9:  93.4%  (4312 chars)
  candidate 10:  95.6%  (4344 chars)


In [26]:
baseline_instructions = prog.analyze.signature.instructions
optimized_instructions = gepa_prog.analyze.signature.instructions
print(
    f"Instructions: {len(baseline_instructions)} characters before, "
    f"{len(optimized_instructions)} after\n"
)
print(optimized_instructions)

Instructions: 2570 characters before, 3977 after

Decide whether a message is a self-contained Caltrain schedule lookup and, if it is, extract the route and the departure time.

## is_schedule_question
True when the message asks which trains run, or when one can go or catch a train, from one specific Caltrain station to another specific, different Caltrain station, optionally at or around a departure time. Otherwise false.

Phrasing does not matter. These are all valid lookups as long as both stations are named and the direction is clear:
- Reversed word order: "I need to get to B from A", "to B from A", "get me to B, leaving from A".
- Conversational or first-person phrasing: "I'm at A; when can I head to B?", "I'm at A, how do I get to B?", "what's the next one?".
- Requests for the next train, with no time given.
- A bare "A to B" with a time, including relative times ("A to B in 20 minutes").
Work out which station is the origin and which is the destination from words like "from", 

# Re-evaluate the saved state

Build a fresh program and load the JSON state the way the bot does, then evaluate that. Two runs show
the run-to-run variation.

In [27]:
optimized_prog = CaltrainScheduleHelper(stations)
optimized_prog.load(str(PROGRAM_PATH))
assert optimized_prog.analyze.signature.instructions == optimized_instructions

In [28]:
optimized_runs = [evaluate(optimized_prog, f"optimized, run {run}") for run in (1, 2)]

2026/11/17 22:03:55 INFO dspy.evaluate.evaluate: Average Metric: 86.0 / 88 (97.7%)
2026/11/17 22:04:04 INFO dspy.evaluate.evaluate: Average Metric: 89.0 / 91 (97.8%)


== optimized, run 1 ==
     train:  97.7%  (86/88 fully correct)
validation:  97.8%  (89/91 fully correct)
    merged:  97.8%  (175/179 fully correct)
LLM/parse errors: 0; misses: 4
  MISS 0.00 [train / unsupported_last_train] 'How late is the final departure from Palo Alto to San Francisco tonight?'
       expected:  is_schedule_question=False with every other output null
       predicted: departure_station='palo alto', arrival_station='san francisco', departure_time=Monday 2026-08-24T15:00
  MISS 0.00 [train / unsupported_ambiguous_direction] "What's the next train between Palo Alto and Redwood City?"
       expected:  is_schedule_question=False with every other output null
       predicted: departure_station='palo alto', arrival_station='redwood city', departure_time=Monday 2026-08-24T15:00
  MISS 0.00 [validation / multiple_misspellings] 'Sunnyvle to Dirdon at 6:12pm today.'
       expected:  is_schedule_question=True, departure_station='sunnyvale', arrival_station='san jose dirido

2026/11/17 22:04:14 INFO dspy.evaluate.evaluate: Average Metric: 86.0 / 88 (97.7%)
2026/11/17 22:04:23 INFO dspy.evaluate.evaluate: Average Metric: 89.0 / 91 (97.8%)


== optimized, run 2 ==
     train:  97.7%  (86/88 fully correct)
validation:  97.8%  (89/91 fully correct)
    merged:  97.8%  (175/179 fully correct)
LLM/parse errors: 0; misses: 4
  MISS 0.00 [train / unsupported_last_train] 'How late is the final departure from Palo Alto to San Francisco tonight?'
       expected:  is_schedule_question=False with every other output null
       predicted: departure_station='palo alto', arrival_station='san francisco', departure_time=Monday 2026-08-24T15:00
  MISS 0.00 [train / unsupported_ambiguous_direction] "What's the next train between Palo Alto and Redwood City?"
       expected:  is_schedule_question=False with every other output null
       predicted: departure_station='palo alto', arrival_station='redwood city', departure_time=Monday 2026-08-24T15:00
  MISS 0.00 [validation / multiple_misspellings] 'Sunnyvle to Dirdon at 6:12pm today.'
       expected:  is_schedule_question=True, departure_station='sunnyvale', arrival_station='san jose dirido

In [29]:
optimized_latency = measure_latency(optimized_prog, "optimized")

optimized: 30 sequential questions, latency median 0.83s, mean 0.87s, max 1.39s; avg tokens per call: prompt 3787, completion 96


# Summary

In [30]:
spend_log["total"] = spent_usd()
time.sleep(60)  # the key's usage figure lags behind the calls
usage_at_end = openrouter_key_usage()

print(f"{'':24}{'train':>8}{'validation':>12}{'merged':>8}")
for run in baseline_runs + optimized_runs:
    print(
        f"{run['label']:24}{run['train']:7.1f}%{run['validation']:11.1f}%{run['merged']:7.1f}%"
        f"   {run['merged_perfect']}/{run['merged_total']} fully correct"
    )

print()
for label, latency, instructions in (
    ("baseline", baseline_latency, baseline_instructions),
    ("optimized", optimized_latency, optimized_instructions),
):
    print(
        f"{label:>9}: median latency {latency['median_s']:.2f}s, "
        f"avg prompt tokens {latency['prompt_tokens']:.0f}, "
        f"instructions {len(instructions)} chars"
    )

print()
print(
    f"GEPA: {gepa_results.total_metric_calls} metric calls, {len(reflection_lm.history)} reflection calls "
    f"({REFLECTION_MODEL}), {gepa_wall_seconds / 60:.1f} min, ${gepa_usd:.2f}"
)
print(
    f"Spend: baseline ${spend_log['baseline']:.4f}, GEPA ${gepa_usd:.4f}, "
    f"final evaluation ${spend_log['total'] - spend_log['gepa']:.4f}, total ${spend_log['total']:.4f}"
)
print(
    f"Key usage cross-check: ${USAGE_AT_START:.4f} at start, ${usage_at_end:.4f} at end, "
    f"difference ${usage_at_end - USAGE_AT_START:.4f}"
)

                           train  validation  merged
baseline, run 1            93.2%       95.6%   94.4%   169/179 fully correct
baseline, run 2            95.5%       94.5%   95.0%   170/179 fully correct
optimized, run 1           97.7%       97.8%   97.8%   175/179 fully correct
optimized, run 2           97.7%       97.8%   97.8%   175/179 fully correct

 baseline: median latency 0.86s, avg prompt tokens 3278, instructions 2570 chars
optimized: median latency 0.83s, avg prompt tokens 3787, instructions 3977 chars

GEPA: 1505 metric calls, 12 reflection calls (anthropic/claude-sonnet-5.5), 4.9 min, $0.93
Spend: baseline $0.1459, GEPA $0.9304, final evaluation $0.1653, total $1.2415
Key usage cross-check: $7.7832 at start, $9.0084 at end, difference $1.2252


Keep the saved prompt only if it earns its place: an optimized prompt that does worse on validation
than the unoptimized one is deleted, so the bot falls back to the built-in instructions.

In [31]:
baseline_validation = statistics.mean(run["validation"] for run in baseline_runs)
optimized_validation = statistics.mean(run["validation"] for run in optimized_runs)
baseline_train = statistics.mean(run["train"] for run in baseline_runs)
optimized_train = statistics.mean(run["train"] for run in optimized_runs)
print(
    f"Mean validation score: baseline {baseline_validation:.1f}%, optimized {optimized_validation:.1f}%"
)
print(
    f"Mean train score:      baseline {baseline_train:.1f}%, optimized {optimized_train:.1f}%"
)

if optimized_instructions == baseline_instructions:
    PROGRAM_PATH.unlink()
    print(
        f"GEPA found nothing better than the built-in instructions: deleted {PROGRAM_PATH}."
    )
elif optimized_validation < baseline_validation:
    PROGRAM_PATH.unlink()
    print(f"The optimized prompt is WORSE on validation: deleted {PROGRAM_PATH}.")
elif optimized_validation == baseline_validation:
    print(
        f"The optimized prompt is NOT better on validation; {PROGRAM_PATH} was kept but adds nothing."
    )
else:
    print(f"The optimized prompt is better on validation: kept {PROGRAM_PATH}.")

Mean validation score: baseline 95.1%, optimized 97.8%
Mean train score:      baseline 94.3%, optimized 97.7%
The optimized prompt is better on validation: kept ../data/schedule_question_program.json.
